# TITLE: Shared fitting of spectral channel light curves

In [ ]:
from eureka.lib import multSpec

In [ ]:
## USER INPUTS ##

optimizer_inputs_file = 'optimizer_inputs.txt'

loc_sci_list = [
"/home/Data/HST/WFC3/HD86226c/Visit04-07",
"/home/Data/HST/WFC3/HD86226c/Visit08-11",
"/home/Data/HST/WFC3/HD86226c/Visit12-15",
# "/home/Data/HST/WFC3/HD86226c/Visit24-27",
"/home/Data/HST/WFC3/HD86226c/Visit28-31"
# "/home/Data/HST/WFC3/HD86226c/Visit32-35",
# "/home/Data/HST/WFC3/HD86226c/Visit36-39"
]

filepath_best_inputs_list = [
'/home/DataAnalysis/HST/HD86226c/Optimized/Visit04-07/Optimized_2024-03-20_HD86226c_run1/',
'/home/DataAnalysis/HST/HD86226c/Optimized/Visit08-11/Optimized_2024-03-20_HD86226c_run1/',
'/home/DataAnalysis/HST/HD86226c/Optimized/Visit12-15/Optimized_2024-03-20_HD86226c_run1/',
# '/home/DataAnalysis/HST/HD86226c/Optimized/Visit24-27/Optimized_2024-03-20_HD86226c_run1/',
'/home/DataAnalysis/HST/HD86226c/Optimized/Visit28-31/Optimized_2024-03-20_HD86226c_run1/'
# '/home/DataAnalysis/HST/HD86226c/Optimized/Visit32-35/Optimized_2024-03-20_HD86226c_run1/',
# '/home/DataAnalysis/HST/HD86226c/Optimized/Visit36-39/Optimized_2024-03-20_HD86226c_run1/'
]                            

# Define the list of lists for transits to mask
# transits_to_mask = [[2, 5] for _ in range(27)]
transits_to_mask = [[] for _ in range(27)]
transits_to_mask[1] = [3]
# transits_to_mask = []  # None


# run_dir = '/home/DataAnalysis/HST/HD86226c/JointSpec/JointSpec_2024-03-23_HD86226c_run1/'
# eventlabel = 'HD86226c'

In [ ]:
# Run multSpec Stages 4-6
eventlabel, run_dir = multSpec.Stage4 (optimizer_inputs_file, loc_sci_list, filepath_best_inputs_list)
multSpec.Stage5(eventlabel, run_dir, transits_to_mask)
multSpec.Stage6(eventlabel, run_dir)

In [ ]:
import os
import glob
import re
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go

def multSpec_S6(eventlabel, run_dir):

    base_path = run_dir

    def find_log_files(base_path, eventlabel):
        """Search for .log files in the specified base_path."""
        return glob.glob(f"{base_path}**/S5_{eventlabel}.log", recursive=True)

    def extract_data_from_file(file_path):
        """Extract the necessary data from each log file."""
        with open(file_path, 'r') as file:
            content = file.read()
            bandpass_matches = re.findall(r'Bandpass \d+ = (\d+\.\d+) - (\d+\.\d+)', content)
            emcee_matches = re.findall(r'EMCEE RESULTS:\nrp: (\d+\.\d+) \(\+([-\d.e]+), -([-\d.e]+)\)', content)

            data = []
            for bandpass in bandpass_matches:
                wave_lo, wave_hi = map(float, bandpass)
                wave = (wave_lo + wave_hi) / 2
                if emcee_matches:
                    rp, err_pos, err_neg = map(float, emcee_matches[0])
                    rp_ppm = (rp ** 2) * 1e6  # Square rp to get the flux value, then convert to ppm
                    err_pos_ppm = (((rp + err_pos) ** 2) - (rp ** 2)) * 1e6  # Convert positive error to ppm
                    err_neg_ppm = abs((((rp - err_neg) ** 2) - (rp ** 2)) * 1e6)  # Convert negative error to ppm
                    # rp_ppm, err_pos_ppm, err_neg_ppm = rp, err_pos, err_neg
                    data.append((wave, wave_lo, wave_hi, rp_ppm, err_pos_ppm, err_neg_ppm))
                    print("Data extracted from EMCEE file.")

            if not data:
                print("No EMCEE data found.")
            return data

    def plot_data(data, eventlabel):
        """
        Plot data with both x and y error bars, centering the y-axis data around 0.

        Parameters:
        - data: A list of tuples, each containing (wave, wave_lo, wave_hi, rp_ppm, err_pos_ppm, err_neg_ppm).
        """
        wave_list, wave_low_list, wave_high_list, rp_list, err_pos_list, err_neg_list = zip(*data)
        
        # Calculate the mean of rp_ppm values and subtract it from each rp_ppm to center around 0
        # mean_rp = np.mean(rp_list)
        mean_rp = 0
        adjusted_rp_list = [rp - mean_rp for rp in rp_list]
        
        xerrors = [[wave - low for wave, low in zip(wave_list, wave_low_list)], 
                [high - wave for wave, high in zip(wave_list, wave_high_list)]]

        plt.figure(figsize=(16, 4))
        plt.errorbar(wave_list, adjusted_rp_list, xerr=xerrors, yerr=[err_neg_list, err_pos_list], fmt='o', capsize=5, 
                    ecolor='darkorange', marker='s', mfc='skyblue', mec='darkblue', 
                    linestyle='None', markersize=6, elinewidth=2)
        
        plt.xlabel('Wavelength (μm)', fontsize=14)
        plt.ylabel('Transit Depth (ppm)', fontsize=14)
        # plt.ylabel('Relative Transit Depth (ppm)', fontsize=14)
        plt.title(eventlabel, fontsize=18)

        plt.grid(True, which='both', linestyle='--', linewidth=0.5)
        plt.minorticks_on()

        plt.xticks(fontsize=12)
        plt.yticks(fontsize=12)
        plt.tight_layout()
        plt.xlim(1.1, 1.7)

        save_dir = base_path + 'S6/'
        if not os.path.exists(save_dir):
            os.makedirs(save_dir)

        plt.savefig(save_dir + 'TransmissionSpectra_' + eventlabel + '_JointSpec.pdf')

        plt.show()

    def plot_data_interactive(data, eventlabel):
        """
        Plot data interactively with both x and y error bars, centering the y-axis data around 0.
        """
        wave_list, wave_low_list, wave_high_list, rp_list, err_pos_list, err_neg_list = zip(*data)

        # Error in wavelength (x-axis)
        x_errors = [[wave - low for wave, low in zip(wave_list, wave_low_list)], 
                    [high - wave for wave, high in zip(wave_list, wave_high_list)]]

        # Error in transit depth (y-axis)
        y_errors = [err_neg_list, err_pos_list]

        fig = go.Figure()

        for x, y, x_err, y_err in zip(wave_list, rp_list, zip(*x_errors), zip(*y_errors)):
            fig.add_trace(go.Scatter(
                x=[x],
                y=[y],
                error_x=dict(type='data', array=[x_err[1]], arrayminus=[x_err[0]]),
                error_y=dict(type='data', array=[y_err[1]], arrayminus=[y_err[0]]),
                mode='markers',
                marker=dict(size=10, color='skyblue', line=dict(width=2, color='DarkSlateGrey')),
                name=''
            ))

        fig.update_layout(
            title=eventlabel,
            xaxis_title='Wavelength (μm)',
            yaxis_title='Transit Depth (ppm)',
            xaxis=dict(range=[1.1, 1.7]),
            template='plotly_white'
        )

        fig.show()

    # You would call plot_data_interactive in your main function instead of plot_data


    def save_data_to_file(data, file_path):
        """Write the data to a .txt file."""
        with open(file_path, 'w') as f:
            f.write("wave wave_lo wave_hi rp_ppm err_pos_ppm err_neg_ppm\n")
            for item in data:
                f.write(" ".join(map(str, item)) + "\n")

    # def main():
    #     log_files = find_log_files(base_path, eventlabel)
    #     all_data = []

    #     for file_path in log_files:
    #         data = extract_data_from_file(file_path)
    #         all_data.extend(data)

    #     if all_data:
    #         # Convert list to numpy array for easier sorting and manipulation
    #         all_data_array = np.array(all_data, dtype=float)
    #         # Sort by wavelength
    #         sorted_all_data = sorted(all_data_array, key=lambda x: x[0])
            
    #         # Identify unique wavelengths and exclude the last two
    #         unique_wavelengths = sorted(set(item[0] for item in sorted_all_data))
    #         if len(unique_wavelengths) > 1:
    #             # Exclude the last two unique wavelengths
    #             wavelengths_to_keep = unique_wavelengths[:-1]
    #             filtered_data = [item for item in sorted_all_data if item[0] in wavelengths_to_keep]
    #         else:
    #             filtered_data = sorted_all_data
            
    #         # Proceed with filtered data
    #         rp_list = [item[3] for item in filtered_data]  # Extracting rp_ppm values
    #         mean_rp_ppm = np.mean(rp_list)
    #         adjusted_rp_list = [(item[0], item[1], item[2], item[3] - mean_rp_ppm, item[4], item[5]) for item in filtered_data]
            
    #         plot_data(adjusted_rp_list, eventlabel)
    #         # Assuming you want to save the adjusted data to a file
    #         save_data_to_file(adjusted_rp_list, 'AdjustedTransmissionSpectra_' + eventlabel + '.txt')
    #         print("Data plotted and adjusted data saved, excluding the last two channels.")
    #     else:
    #         print("No data found.")

    def main():
        log_files = find_log_files(base_path, eventlabel)
        all_data = []

        for file_path in log_files:
            data = extract_data_from_file(file_path)
            all_data.extend(data)

        if all_data:
            # Convert list to numpy array for easier sorting and manipulation
            all_data_array = np.array(all_data, dtype=float)
            sorted_all_data = sorted(all_data_array, key=lambda x: x[0])  # Sort by wavelength

            sorted_all_data = sorted_all_data[1:]

            plot_data(sorted_all_data, eventlabel)
            save_dir = base_path + 'S6/'
            if not os.path.exists(save_dir):
                os.makedirs(save_dir)
            save_data_to_file(sorted_all_data, save_dir + 'TransmissionSpectra_' + eventlabel + '_JointSpec.txt')
            print("Data plotted and saved.")
            plot_data_interactive(sorted_all_data, eventlabel)

        else:
            print("No data found.")

    # if __name__ == "__main__":
    main()


In [ ]:
# multSpec.Stage6(eventlabel, run_dir)
multSpec_S6(eventlabel, run_dir)